# ENGE817 — Pilot Study
## Network Traffic Analysis of Smart Solar IoT Inverters.



```
  inverter_simulator.py  ──── Modbus/TCP ────▶  monitoring_client.py
   = the "smart inverter"      port 502          = the "monitoring app"
            │                                             │
            └─────────────────┬───────────────────────────┘
                              ▼
                   [ Wireshark records it ]
                              ▼
                     pilot_capture.pcap
                              ▼
                       analysis
```


---
## Cell 1 — Installing Wireshark and the Python libraries

In [ ]:
# Install Wireshark (tshark) and the Python libraries
import subprocess, sys, os

print("Installing Wireshark (tshark)... please wait ~60 seconds\n")
os.environ["DEBIAN_FRONTEND"] = "noninteractive"
subprocess.run("apt-get update -qq", shell=True, capture_output=True)
subprocess.run("apt-get install -y -qq tshark tcpdump", shell=True, capture_output=True)

print("Installing Python libraries...\n")
subprocess.run(sys.executable + " -m pip install -q 'pymodbus==3.6.9' scapy",
               shell=True, capture_output=True)

ok = True
try:
    tsv = subprocess.run("tshark --version", shell=True, capture_output=True,
                         text=True).stdout.split("\n")[0]
    if not tsv.strip():
        ok = False
        tsv = "NOT FOUND"
except Exception:
    ok = False
    tsv = "NOT FOUND"

print("=" * 64)
print("TOOL VERSIONS (record these for Section 4.3 of your report)")
print("=" * 64)
print("Wireshark :", tsv)
try:
    import pymodbus, scapy
    print("pymodbus  :", pymodbus.__version__, "(pinned for reproducibility)")
    print("scapy     :", scapy.__version__)
except Exception as e:
    ok = False
    print("Python library import FAILED:", e)
print("python    :", sys.version.split()[0])

print("\nCapture interfaces Wireshark can see:")
ifaces = subprocess.run("tshark -D 2>/dev/null", shell=True,
                        capture_output=True, text=True).stdout.strip()
print(ifaces if ifaces else "   (none listed)")
print("=" * 64)
print("\nINSTALL COMPLETE" if ok else "\nPROBLEM: re-run this cell.")

---
## Cell 2 — Creating the inverter simulator.

This writes the program that pretends to be a smart solar inverter. It serves four
telemetry readings over Modbus/TCP, exactly as a real SunSpec-compliant inverter does:

| Register | Meaning | Units |
|---|---|---|
| 0 | AC power output | Watts |
| 1 | DC voltage | Volts × 10 |
| 2 | Grid frequency | Hz × 100 |
| 3 | Energy today | kWh × 10 |



In [ ]:
%%writefile inverter_simulator.py
"""Simulates a smart solar inverter serving telemetry over Modbus/TCP -
the same UNENCRYPTED, UNAUTHENTICATED protocol used by many real inverters."""
import random
import threading
import time
from pymodbus.datastore import (ModbusSequentialDataBlock, ModbusServerContext,
                                ModbusSlaveContext)
from pymodbus.server import StartTcpServer

store = ModbusSlaveContext(hr=ModbusSequentialDataBlock(0, [0] * 100))
context = ModbusServerContext(slaves=store, single=True)


def update_readings():
    """Every second, write fresh telemetry into the holding registers."""
    while True:
        ac_power = random.randint(1500, 5000)
        dc_voltage = int(random.uniform(380, 420) * 10)
        frequency = int(random.uniform(49.8, 50.2) * 100)
        energy = int(random.uniform(10, 30) * 10)
        context[0].setValues(3, 0, [ac_power, dc_voltage, frequency, energy])
        print("[inverter] P=" + str(ac_power) + "W  Vdc=" + str(dc_voltage / 10)
              + "V  f=" + str(frequency / 100) + "Hz  E=" + str(energy / 10) + "kWh",
              flush=True)
        time.sleep(1)


if __name__ == "__main__":
    print("Smart-inverter simulator listening on 127.0.0.1:502 (Modbus/TCP)", flush=True)
    time.sleep(0.3)
    threading.Thread(target=update_readings, daemon=True).start()
    StartTcpServer(context=context, address=("127.0.0.1", 502))

---
## Cell 3 — Creating the monitoring client.

In [ ]:
%%writefile monitoring_client.py
"""Simulates the vendor monitoring platform polling the inverter once per second."""
import time
from pymodbus.client import ModbusTcpClient

client = ModbusTcpClient("127.0.0.1", port=502)
client.connect()
print("Monitoring client connected to inverter. Polling telemetry...", flush=True)

try:
    while True:
        rr = client.read_holding_registers(address=0, count=4)
        if not rr.isError():
            p, v, f, e = rr.registers
            print("[monitor] read -> P=" + str(p) + "W  Vdc=" + str(v / 10)
                  + "V  f=" + str(f / 100) + "Hz  E=" + str(e / 10) + "kWh", flush=True)
        else:
            print("[monitor] read error:", rr, flush=True)
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    client.close()

---
## Cell 4 — Create your analysis script

This writes the Python program that reads your capture file and answers the research
question automatically: which protocol, is it encrypted, is it authenticated, and what
can an eavesdropper actually recover.

In [ ]:
%%writefile parse_capture.py
"""Reads a .pcap capture and analyses smart-inverter telemetry traffic.
Answers: what protocol? is it encrypted? what can an eavesdropper read?"""
import sys
from scapy.all import rdpcap, TCP, Raw

MODBUS_FUNCS = {1: "Read Coils", 2: "Read Discrete Inputs",
                3: "Read Holding Registers", 4: "Read Input Registers",
                5: "Write Single Coil", 6: "Write Single Register",
                15: "Write Multiple Coils", 16: "Write Multiple Registers"}


def analyse(pcap_file):
    packets = rdpcap(pcap_file)
    print("Loaded " + str(len(packets)) + " packets from " + pcap_file + "\n")
    modbus_packets = 0
    request_count = 0
    response_count = 0
    values_seen = []

    for pkt in packets:
        if TCP in pkt and Raw in pkt:
            payload = bytes(pkt[Raw].load)
            dport = pkt[TCP].dport
            sport = pkt[TCP].sport
            if 502 in (dport, sport) and len(payload) >= 8:
                modbus_packets += 1
                func = payload[7]
                fname = MODBUS_FUNCS.get(func, "Function " + str(func))
                if dport == 502:
                    request_count += 1
                    print("  REQUEST  (monitor -> inverter)    " + fname)
                else:
                    response_count += 1
                    print("  RESPONSE (inverter -> monitor)    " + fname)
                    if func == 3 and len(payload) >= 9:
                        bc = payload[8]
                        data = payload[9:9 + bc]
                        regs = [int.from_bytes(data[i:i + 2], "big")
                                for i in range(0, len(data), 2)]
                        values_seen.append(regs)

    print("\n" + "=" * 60)
    print("ANALYSIS SUMMARY")
    print("=" * 60)
    print("Modbus/TCP packets found : " + str(modbus_packets))
    print("  Requests (polls)       : " + str(request_count))
    print("  Responses (telemetry)  : " + str(response_count))

    if modbus_packets:
        print("\nProtocol detected        : Modbus/TCP (SunSpec-style telemetry)")
        print("Transport security       : NONE  <-- payload is PLAIN TEXT")
        print("Authentication           : NONE  <-- no login/keys on the wire")
        print("\nData readable by an eavesdropper (decoded from responses):")
        for regs in values_seen[:5]:
            if len(regs) >= 4:
                print("   AC power=" + str(regs[0]) + "W  DC volts="
                      + str(regs[1] / 10) + "V  freq=" + str(regs[2] / 100)
                      + "Hz  energy=" + str(regs[3] / 10) + "kWh")
        print("\nFINDING: Modbus/TCP provides no encryption or authentication, so all")
        print("inverter telemetry is exposed and commands could be forged or replayed.")
    else:
        print("\nNo Modbus packets found - check the capture step.")


if __name__ == "__main__":
    analyse(sys.argv[1] if len(sys.argv) > 1 else "pilot_capture.pcap")

---
#  Cell 5 — THE PILOT.

In [ ]:
# ============================================================
#  RUN THE PILOT STUDY
# ============================================================
import subprocess, time, os, datetime

CAPTURE_SECONDS = 60          # increase for a longer run
PCAP = "pilot_capture.pcap"

for f in (PCAP, "sim.log", "client.log", "capture.log"):
    if os.path.exists(f):
        os.remove(f)

run_started = datetime.datetime.now()
print("Pilot run started:", run_started.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 64)

print("[1/5] Starting the inverter simulator...")
sim = subprocess.Popen(["python3", "inverter_simulator.py"],
                       stdout=open("sim.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)

if sim.poll() is not None:
    print("\nERROR: the simulator stopped unexpectedly. Its log says:\n")
    print(open("sim.log").read()[:1500])
    raise SystemExit("Fix the error above, then re-run Cell 2 and this cell.")

print("[2/5] Starting Wireshark (tshark) capture on loopback...")
cap = subprocess.Popen(["tshark", "-i", "lo", "-f", "tcp port 502",
                        "-w", PCAP, "-a", "duration:" + str(CAPTURE_SECONDS)],
                       stdout=open("capture.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

print("[3/5] Starting the monitoring platform...")
cli = subprocess.Popen(["python3", "monitoring_client.py"],
                       stdout=open("client.log", "w"), stderr=subprocess.STDOUT)

print("[4/5] Recording telemetry for " + str(CAPTURE_SECONDS)
      + " seconds. Please wait...")
try:
    cap.wait(timeout=CAPTURE_SECONDS + 90)
except Exception:
    cap.kill()

print("[5/5] Stopping all processes...")
for p in (cli, sim):
    p.terminate()
    try:
        p.wait(timeout=5)
    except Exception:
        p.kill()

print("=" * 64)
size = os.path.getsize(PCAP) if os.path.exists(PCAP) else 0

if size > 100:
    print("PILOT COMPLETE")
    print("Capture file :", PCAP, "(" + str(size) + " bytes)")
    print("Duration     :", CAPTURE_SECONDS, "seconds")
    print("Finished     :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 64)
    print("\n--- Inverter was producing (first readings) ---")
    print("".join(open("sim.log").readlines()[:4]))
    print("--- Monitoring platform was receiving (first reads) ---")
    print("".join(open("client.log").readlines()[:4]))
else:
    print("PROBLEM: the capture file is empty.")
    print("\ncapture.log says:")
    print(open("capture.log").read()[:800] if os.path.exists("capture.log") else "(none)")
    print("\nclient.log says:")
    print(open("client.log").read()[:800] if os.path.exists("client.log") else "(none)")
    print("\nTry: Runtime -> Restart session, then run all cells again from Cell 1.")

---
## Cell 6 — The capturing conversation.

In [ ]:
import subprocess

print("=" * 78)
print("FIGURE 3: Captured Modbus/TCP telemetry exchange (Wireshark packet list)")
print("=" * 78)

out = subprocess.run("tshark -r pilot_capture.pcap -Y mbtcp 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout
lines = [l for l in out.split("\n") if l.strip()]
for line in lines[:14]:
    print(line)
print("\n(" + str(len(lines)) + " Modbus packets in total)")

---
## Cell 7 — The protocol decoded.

The Wireshark detail pane. Wireshark recognises the traffic as Modbus and breaks
it into named fields — **and it reads the actual sensor values straight off the wire**.

In [ ]:
import subprocess

print("=" * 78)
print("FIGURE 4: Modbus/TCP response dissected - register values in plaintext")
print("=" * 78)

out = subprocess.run(
    'tshark -r pilot_capture.pcap -Y "modbus.func_code==3 && tcp.srcport==502" '
    '-V 2>/dev/null', shell=True, capture_output=True, text=True).stdout

KEYS = ("Transaction Identifier", "Protocol Identifier", "Unit Identifier",
        "Function Code", "Byte Count", "Register 0", "Register 1",
        "Register 2", "Register 3")
shown = 0
for line in out.split("\n"):
    if any(k in line for k in KEYS) and "Register Value" not in line:
        print(line.rstrip())
        shown += 1
        if shown >= 9:
            break
print("\n(One complete telemetry response, as Wireshark dissects it.)")

---
## Cell 8 — The raw bytes.

In [ ]:
import subprocess

print("=" * 78)
print("FIGURE 5: Raw bytes of a telemetry response - no encryption present")
print("=" * 78)

out = subprocess.run(
    'tshark -r pilot_capture.pcap -Y "modbus.func_code==3 && tcp.srcport==502" '
    '-x 2>/dev/null', shell=True, capture_output=True, text=True).stdout
for line in out.split("\n")[:7]:
    print(line.rstrip())
print("\nThe Modbus payload sits at the end of the final rows:")
print("  function code 03, byte count 08, then the four readings - all in the clear.")

---
## Cell 9 — What an eavesdropper actually recovers.

In [ ]:
import subprocess

print("=" * 78)
print("FIGURE 6: Household telemetry recovered by a passive observer")
print("=" * 78)

out = subprocess.run(
    'tshark -r pilot_capture.pcap -Y "modbus.func_code==3 && tcp.srcport==502" '
    '-T fields -e modbus.regval_uint16',
    shell=True, capture_output=True, text=True).stdout.strip()
rows = [r for r in out.split("\n") if r.strip()]

header = ("Reading".ljust(9) + "AC Power".rjust(12) + "DC Voltage".rjust(14)
          + "Frequency".rjust(13) + "Energy today".rjust(15))
print("\n" + header)
print("-" * 63)
for i, line in enumerate(rows[:10], 1):
    v = line.split(",")
    if len(v) >= 4:
        print(str(i).ljust(9)
              + (v[0] + " W").rjust(12)
              + ("{:.1f} V".format(int(v[1]) / 10)).rjust(14)
              + ("{:.2f} Hz".format(int(v[2]) / 100)).rjust(13)
              + ("{:.1f} kWh".format(int(v[3]) / 10)).rjust(15))
print("-" * 63)
print("\nTotal telemetry responses intercepted: " + str(len(rows)))
print("None of this required any key, password, or decryption.")

---
## Cell 10 — Automated analysis.

This runs your own Python analysis script over the capture — the parsing step your
research design promised. It independently confirms the protocol, the security posture,
and the exposed data.

In [ ]:
print("=" * 78)
print("FIGURE 7: Automated packet analysis output (parse_capture.py)")
print("=" * 78)
!python3 parse_capture.py pilot_capture.pcap

---
## Cell 11 — The control comparison (TLS).

So far you have shown Modbus is unprotected. On its own that is one observation. The
**comparison** turns it into an analytical finding.

This captures ordinary HTTPS traffic — which uses **TLS**, the same protection IEEE
2030.5 requires — and shows what protected traffic looks like.

**Compare the two outputs below.** Same machine, same network, two protocols: one hands
over every reading, the other gives up nothing.

In [ ]:
import subprocess, time, os

if os.path.exists("tls_capture.pcap"):
    os.remove("tls_capture.pcap")

print("Capturing encrypted HTTPS/TLS traffic for comparison...\n")
cap = subprocess.Popen(["tshark", "-i", "any", "-f", "tcp port 443",
                        "-w", "tls_capture.pcap", "-a", "duration:15"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
subprocess.run("curl -s https://example.com -o /dev/null", shell=True)
subprocess.run("curl -s https://www.wireshark.org -o /dev/null", shell=True)
try:
    cap.wait(timeout=60)
except Exception:
    cap.kill()

def show(cmd, n=7):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout
    for line in out.split("\n")[:n]:
        print(line.rstrip())

print("=" * 78)
print("FIGURE 8a: TLS-protected traffic - payload is unreadable ciphertext")
print("=" * 78)
show('tshark -r tls_capture.pcap -Y "tls.app_data" -x 2>/dev/null')
print("\n   ^ random-looking bytes. Nothing is recoverable without the keys.\n")

print("=" * 78)
print("FIGURE 8b: CONTRAST - the same byte view for Modbus telemetry")
print("=" * 78)
show('tshark -r pilot_capture.pcap -Y "modbus.func_code==3 && tcp.srcport==502" '
     '-x 2>/dev/null')
print("\n   ^ structured, readable, and decoded straight into sensor values.")

---
## Cell 12 — Results table.

In [ ]:
import subprocess, os, datetime, platform

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

def count(filter_expr=""):
    cmd = "tshark -r pilot_capture.pcap"
    if filter_expr:
        cmd += ' -Y "' + filter_expr + '"'
    cmd += " 2>/dev/null"
    return len([l for l in sh(cmd).split("\n") if l.strip()])

total = count()
modbus = count("mbtcp")
queries = count("mbtcp && tcp.dstport==502")
responses = count("mbtcp && tcp.srcport==502")
tsv = sh("tshark --version").split("\n")[0]
secs = CAPTURE_SECONDS if "CAPTURE_SECONDS" in dir() else 60

print("=" * 72)
print("PILOT STUDY RESULTS - copy this into your report")
print("=" * 72)
rows = [
    ("Date of capture", datetime.date.today().strftime("%d %B %Y")),
    ("Environment", "Google Colab (Linux VM), loopback interface"),
    ("Python version", platform.python_version()),
    ("Capture tool", tsv),
    ("Capture duration", str(secs) + " seconds"),
    ("Capture file size", str(os.path.getsize("pilot_capture.pcap")) + " bytes"),
    ("Total packets captured", total),
    ("Modbus/TCP packets", modbus),
    ("  Queries (polls)", queries),
    ("  Responses (telemetry)", responses),
    ("Protocol identified", "Modbus/TCP (SunSpec-style), TCP port 502"),
    ("Transport encryption", "None observed"),
    ("Authentication exchange", "None observed"),
    ("Telemetry fields exposed", "AC power, DC voltage, grid frequency, energy"),
    ("Control comparison", "TLS (HTTPS) - payload unreadable"),
]
for k, v in rows:
    print(str(k).ljust(28) + " : " + str(v))
print("=" * 72)
print("\nSUMMARY SENTENCE FOR YOUR ABSTRACT:")
print('  "' + str(responses) + " telemetry responses were intercepted over "
      + str(secs) + ' seconds;')
print("   all were transmitted in plaintext with no authentication, exposing")
print('   power, voltage, frequency and energy readings to any passive observer."')

---
## Cell 13 — Downloading evidence.

In [ ]:
import os, shutil

EVIDENCE = "ENGE817_pilot_evidence"
if os.path.exists(EVIDENCE):
    shutil.rmtree(EVIDENCE)
os.makedirs(EVIDENCE)

wanted = ["pilot_capture.pcap", "tls_capture.pcap", "inverter_simulator.py",
          "monitoring_client.py", "parse_capture.py",
          "sim.log", "client.log", "capture.log"]

print("Collecting evidence files:")
for f in wanted:
    if os.path.exists(f):
        shutil.copy(f, EVIDENCE)
        print("   " + f + "  (" + str(os.path.getsize(f)) + " bytes)")

shutil.make_archive(EVIDENCE, "zip", EVIDENCE)
print("\nArchive created: " + EVIDENCE + ".zip ("
      + str(os.path.getsize(EVIDENCE + ".zip")) + " bytes)")

try:
    from google.colab import files
    files.download(EVIDENCE + ".zip")
    print("Download started - check your browser's downloads.")
except Exception as e:
    print("\nNot running in Colab, or download blocked (" + str(e) + ").")
    print("Use the folder icon in the left sidebar to download "
          + EVIDENCE + ".zip manually.")